# Lesson 05 Lab — Diagnosing FP16 Overflow and Gradient Scaling Failures

**Puzzle:** When loss becomes NaN, how do we distinguish forward overflow, backward overflow, and gradient underflow?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A final NaN is the last symptom in a chain, not the diagnosis. FP16 can overflow during forward, overflow after loss scaling during backward, or silently round tiny gradients to zero. Each failure calls for a different response, so the first bad tensor must be located before changing the scaler.


## 0. Predict before running

1. Predict which combinations of gradient magnitude and loss scale become zero, finite, or infinite in FP16.
2. Explain why loss scaling can rescue underflow but cannot repair a forward activation that is already Inf.
3. Choose probe locations that distinguish forward, scaled-backward, unscaled-gradient, and parameter corruption.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Diagnose four checkpoints: forward outputs, scaled loss/gradients, unscaled gradients, and post-step parameters. A final NaN has already discarded the location of the first failure.

- Overflow creates Inf before it becomes NaN in later arithmetic.
- Underflow silently rounds small gradients to zero.
- Loss scaling moves gradients into a representable interval but cannot repair an already-overflowed forward pass.


## 2. Derive the mechanism

FP16 normal values end near `6.55e4`; very small values enter a sparse subnormal region and can become zero. Loss scaling shifts gradient magnitudes upward during storage, but unscaling must happen before clipping and parameter updates.

With loss scale S, an exact gradient g is represented during backward as `Sg`. If g is smaller than the FP16 subnormal range, choosing a moderate S can move it onto the representable grid; unscale later restores its mathematical magnitude in a wider type. If `Sg > 65504`, the scaled gradient becomes Inf. And if a forward value already exceeded 65504, multiplying the loss later cannot reconstruct the discarded information.

This creates a feasible interval for S: large enough that important small gradients survive, but small enough that the largest scaled gradient remains finite. Dynamic scaling searches that interval through observed overflow. It does not guarantee that every tiny gradient is preserved or that the forward pass is stable.

### Mechanism at a glance

```mermaid
flowchart TD
  A["Non-finite loss or bad update"] --> B{"first bad tensor?"}
  B -->|"forward activation"| C["Change forward dtype,<br/>normalization, or input range"]
  B -->|"scaled gradient is Inf"| D["Lower scale and skip step"]
  B -->|"tiny gradient became zero"| E["Raise scale or use wider dtype"]
  B -->|"after optimizer"| F["Inspect unscale, clipping,<br/>optimizer state, and LR"]
  C --> R["Replay the same batch"]
  D --> R
  E --> R
  F --> R
```

### Walk it step by step

1. **Locate the first bad stage.** Check forward activations, the scaled loss, scaled gradients, unscaled gradients, and parameters in that order.
2. **Classify the symptom.** Inf indicates overflow; excessive zeros can indicate underflow even though every value is finite.
3. **Apply the matching intervention.** Lower the scale for scaled-gradient overflow, raise it for underflow, or change the forward dtype for activation overflow.
4. **Replay the same batch.** A diagnosis is useful only when the intervention removes the original first failure without creating a new one.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "05-fp16-overflow"
device = require_cuda()
torch.manual_seed(2026 + 5)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | FP16 casting of four gradient magnitudes with scale 1 |
| Candidate | the same magnitudes multiplied by scales 256 and 65536 |
| Held constant | tensor size, dtype, GPU, values within each magnitude group |
| Measurements | zero fraction, finite fraction, Inf fraction, plus a separate forward-overflow probe |
| Evidence | `pytorch-gpu` |

**Experiment:** Sweep synthetic gradient magnitudes and loss scales in FP16 on CUDA, counting finite, infinite, and zero gradient values.


## 5. Read the experiment code

The CUDA sweep crosses both tiny and large magnitudes at several scales and records zero and Inf fractions, making the failure stage observable.

The notebook sweeps a Cartesian product rather than waiting for a random training failure. For every magnitude/scale pair it casts the scaled value to FP16 and counts zero, finite, and infinite entries. A separate `1e5` forward probe establishes that some damage can occur before backward begins.

Because all elements in a row share one magnitude, fractions jump cleanly between zero, finite, and Inf. A real model would produce a distribution, but the synthetic grid makes the representability boundaries easy to see and debug.

Only after these variables match the protocol should the cell be executed.


In [2]:
rows = []
for magnitude in (1e-8, 1e-5, 1.0, 1e3):
    for scale in (1.0, 256.0, 65536.0):
        p = torch.ones(4096, device=device, dtype=torch.float16, requires_grad=True)
        loss = (p.float() * magnitude).sum() * scale
        loss.backward(); g = p.grad
        rows.append({"magnitude": magnitude, "loss_scale": scale,
                     "zero_fraction": round((g == 0).float().mean().item(), 6),
                     "inf_fraction": round(torch.isinf(g).float().mean().item(), 6),
                     "finite_fraction": round(torch.isfinite(g).float().mean().item(), 6)})
forward_overflow = torch.isinf(torch.tensor([1e5], device=device).half()).item()
result = base_result(5, "pytorch-gpu"); result.update({"gradient_sweep": rows,
    "forward_overflow_at_1e5": bool(forward_overflow),
    "conclusion": "Scaling changed gradient representability but could not repair an FP16 value that had already overflowed."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| 1e-8, scale 1: zero fraction | 100.0000% |
| 1e-8, scale 256: zero fraction | 0.0000% |
| 1, scale 65536: Inf fraction | 100.0000% |
| 1000, scale 256: Inf fraction | 100.0000% |
| Forward 1e5 overflowed | yes |


## 7. Interpret rather than merely print

At magnitude `1e-8`, scale 1 rounded every value to zero, while scales 256 and 65536 made all entries finite and non-zero. At magnitude 1, scale 65536 overflowed every value. At magnitude 1000, scale 256 was already too large. The independent forward test confirmed that FP16 `1e5` was non-finite.

The same tool—larger scale—therefore fixes one row and breaks another. That is the central reason GradScaler adapts and skips unsafe optimizer steps. It is also why a scaler change is the wrong fix for forward overflow.

**Inspection rule:** The useful evidence is the first stage where finiteness changes. A final NaN without intermediate checks is not a diagnosis.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Scaling changed gradient representability but could not repair an FP16 value that had already overflowed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:17+00:00",
  "forward_overflow_at_1e5": true,
  "gradient_sweep": [
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale": 1.0,
      "magnitude": 1e-08,
      "zero_fraction": 1.0
    },
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale": 256.0,
      "magnitude": 1e-08,
      "zero_fraction": 0.0
    },
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale": 65536.0,
      "magnitude": 1e-08,
      "zero_fraction": 0.0
    },
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale

## 9. Make the bounded decision

> Place finiteness and zero-rate probes at forward outputs, scaled gradients, unscaled gradients, and parameters before changing the scaler policy.

**Acceptance/rollback:** Log finite/Inf/zero fractions and the current scale. If the forward pass is already non-finite, change the operation or dtype; if only scaled gradients overflow, adjust scale policy.

**Failure analysis:** Looking only at `torch.isfinite(loss)` misses underflow because zeros are finite. Looking only after unscale can hide where overflow began. Logging every tensor is too expensive, so production diagnosis usually places targeted hooks at loss, selected activations, scaled gradients, unscaled gradients, and parameters, then narrows the search.


## 10. Extend the evidence

Instrument a small FP16 network with hooks that report min/max, zero fraction, and finiteness at the four stages. Inject an activation spike and a tiny-gradient layer separately. Verify that lowering the scale helps the first backward-overflow case, raising it helps the underflow case, and neither repairs the injected forward Inf.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
